# Zebrafish Chemical–Gene (STITCH) — Relation-Wise KG Triple Construction

## Purpose

This notebook processes **Chemical–Protein interaction data** from the STITCH database for Zebrafish (*Danio rerio*) and transforms it into standardized relation-wise Knowledge Graph (KG) triples. It includes both the raw STITCH data preprocessing (TSV → pickle) and the final KG triple construction.

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | Setup & Dictionaries | Load gProfiler and PubChem data; build mapping dictionaries |
| 2 | STITCH Raw Preprocessing | Load raw STITCH TSV, clean protein IDs, add metadata, save pickle |
| 3 | KG Triple Construction | Load pickle, clean chemical IDs, map proteins to genes, annotate |
| 4 | Filter, Validate, Deduplicate & Export | Remove unmapped entries, validate, deduplicate, export final CSV |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | PubChem CID (chemical identifier) |
| `relation` | `ChemicalEntity_Gene` |
| `tail` | Gene symbol |
| `head_type` | `ChemicalEntity` |
| `relation_type` | (NaN — no sub-relation) |
| `tail_type` | `Gene` |
| `kg_source` | `STITCH` |
| `head_id_is` | `Pubchem` |
| `tail_id_is` | `NCBI_ID` |
| `head_detail_name` | PubChem IUPAC chemical name |
| `tail_detail_name` | Gene description from gProfiler |
| `species` | `D.rerio` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `7955.protein_chemical.links.v5.0.tsv` | STITCH | Raw chemical–protein interaction file for zebrafish (tax_id: 7955) |
| `gProfiler_drerio_*.csv` | gProfiler | Ensembl protein ID to gene symbol/description mapping |
| `combined_df.pkl` | PubChem (pre-processed) | PubChem compound data with CID, IUPAC names, and SMILES |

---
## Setup

Import required libraries and define all base paths.

In [1]:
import pandas as pd
import numpy as np

In [2]:
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

In [22]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================

# gProfiler mapping file (Ensembl protein ID → gene symbol)
GPROFILER_PATH = f'{your_path_here}stitch/drerio/gProfiler_drerio_4-9-2025_5-19-17 PM.csv'

# PubChem compound data (pre-processed pickle with CID, IUPAC, SMILES)
PUBCHEM_PATH = f'{your_path_here}databases_for_mapping/pubchem/combined_df.pkl'

# Raw STITCH chemical-protein interaction file for zebrafish
STITCH_RAW_TSV_PATH = f'{your_path_here}stitch/drerio/7955.protein_chemical.links.v5.0.tsv'

# Intermediate pickle output (from raw preprocessing)
# STITCH_PICKLE_PATH = f'{your_path_here}/ZEBRAFISH/CHEMICAL_PROTEIN/ZF_C_P_STITCH_Triples.pkl'

# Final output path
OUTPUT_PATH = f'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_ZEBRA_CHEMICALENTITY_GENE.csv'

---
## 1. Load Mapping Dictionaries

### 1a. gProfiler (Ensembl Protein ID → Gene Symbol + Description)

gProfiler provides a mapping from Ensembl protein IDs (e.g., `ENSDARP00000124868`) to gene symbols (e.g., `prf1.2`) and descriptions.

In [6]:
# =============================================================================
# Load gProfiler data and build mapping dictionaries
# =============================================================================

gprofiler_mapping = pd.read_csv(GPROFILER_PATH)
gprofiler_mapping = gprofiler_mapping[~gprofiler_mapping['converted_alias'].isna()]

# Dictionary: Ensembl Protein ID → Gene Symbol
protein_to_gene_dict = dict(zip(gprofiler_mapping['initial_alias'], gprofiler_mapping['name']))

# Dictionary: Gene Symbol → Gene Description (used for tail_detail_name)
gene_to_description_dict = dict(zip(gprofiler_mapping['name'], gprofiler_mapping['description']))

print(f"gProfiler entries loaded: {len(gprofiler_mapping):,}")
print(f"Protein → Gene dictionary size: {len(protein_to_gene_dict):,}")
print(f"Gene → Description dictionary size: {len(gene_to_description_dict):,}")
gprofiler_mapping.head()

gProfiler entries loaded: 44,669
Protein → Gene dictionary size: 24,474
Gene → Description dictionary size: 24,117


,initial_alias,converted_alias,name,description,namespace
0,ENSDARP00000000004,ENSDARP00000000004,slc35a5,solute carrier family 35 member A5 [Source:NCB...,ENSP
1,ENSDARP00000000005,ENSDARP00000000005,ccdc80,coiled-coil domain containing 80 [Source:NCBI ...,ENSP
2,ENSDARP00000000042,ENSDARP00000000042,mcm6l,"MCM6 minichromosome maintenance deficient 6, l...",ENSP
3,ENSDARP00000000042,ENSDARP00000119510,mcm6l,"MCM6 minichromosome maintenance deficient 6, l...",ENSP
4,ENSDARP00000000069,ENSDARP00000000069,nherf1a,NHERF family PDZ scaffold protein 1a [Source:Z...,ENSP


### 1b. PubChem (CID → IUPAC Name)

Load PubChem compound data to build a dictionary mapping CIDs to IUPAC chemical names for annotating the head entity.

In [7]:
# =============================================================================
# Load PubChem compound data and build CID → IUPAC name dictionary
# =============================================================================

pubchem_df = pd.read_pickle(PUBCHEM_PATH)

# Dictionary: PubChem CID → IUPAC Name (used for head_detail_name annotation)
pubchem_cid_to_iupac = dict(zip(pubchem_df['PUBCHEM_COMPOUND_CID'], pubchem_df['PUBCHEM_IUPAC_NAME']))

print(f"PubChem compounds loaded: {len(pubchem_df):,}")
print(f"CID → IUPAC dictionary size: {len(pubchem_cid_to_iupac):,}")

# Free memory
del pubchem_df

PubChem compounds loaded: 119,177,440
CID → IUPAC dictionary size: 119,177,440


---
## 2. STITCH Raw Preprocessing (TSV → Pickle)

This section loads the raw STITCH chemical–protein interaction file, cleans protein identifiers, adds relation metadata, and saves the result as a pickle.

**Processing steps:**
1. Load raw TSV (columns: chemical, protein, combined_score)
2. Rename columns to Head/Tail
3. Add relation metadata (Relation, Relation_Source, head_type, tail_type)
4. Strip species prefix (`7955.`) from protein IDs
5. Remove trailing version numbers from protein IDs
6. Save as pickle

**Note:** This section only needs to be run once to create the pickle. After that, Section 3 can load the pickle directly.

In [8]:
# =============================================================================
# Load raw STITCH chemical-protein interaction file
# =============================================================================
# The file uses whitespace as separator
# Columns: chemical, protein, combined_score

stitch_chem_protein_raw = pd.read_csv(STITCH_RAW_TSV_PATH, sep='\\s')
stitch_chem_protein_raw = stitch_chem_protein_raw.rename(columns={'chemical': 'Head', 'protein': 'Tail'})

print(f"Raw STITCH interactions loaded: {len(stitch_chem_protein_raw):,}")
stitch_chem_protein_raw.head()

/tmp/ipykernel_1661938/757708189.py:7: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  stitch_chem_protein_raw = pd.read_csv(STITCH_RAW_TSV_PATH, sep='\\s')


Raw STITCH interactions loaded: 74,619,878


,Head,Tail,combined_score
0,CIDm91758680,7955.ENSDARP00000068614,219
1,CIDm91758680,7955.ENSDARP00000102799,219
2,CIDm91758680,7955.ENSDARP00000108783,219
3,CIDm91758408,7955.ENSDARP00000006504,160
4,CIDm91758408,7955.ENSDARP00000014299,241


In [9]:
# =============================================================================
# Add relation metadata and clean protein identifiers
# =============================================================================

stitch_chem_protein_raw['Relation'] = 'Chemical_Protein'
stitch_chem_protein_raw['Relation_Source'] = 'STITCH'
stitch_chem_protein_raw['head_type'] = 'Chemical'
stitch_chem_protein_raw['tail_type'] = 'Protein'

# Reorder columns
stitch_chem_protein_raw = stitch_chem_protein_raw[['Head', 'Relation', 'Tail', 'combined_score', 'head_type', 'tail_type', 'Relation_Source']]

# Strip species prefix '7955.' from protein IDs
stitch_chem_protein_raw['Tail'] = stitch_chem_protein_raw['Tail'].str.replace('7955.', '')

# Remove trailing version numbers (e.g., ENSDARP00000068614.1 → ENSDARP00000068614)
stitch_chem_protein_raw['Tail'] = stitch_chem_protein_raw['Tail'].apply(lambda x: x.rsplit('.', 1)[0] if x.count('.') > 1 else x)

print(f"Protein IDs cleaned")
stitch_chem_protein_raw.head()

Protein IDs cleaned


,Head,Relation,Tail,combined_score,head_type,tail_type,Relation_Source
0,CIDm91758680,Chemical_Protein,ENSDARP00000068614,219,Chemical,Protein,STITCH
1,CIDm91758680,Chemical_Protein,ENSDARP00000102799,219,Chemical,Protein,STITCH
2,CIDm91758680,Chemical_Protein,ENSDARP00000108783,219,Chemical,Protein,STITCH
3,CIDm91758408,Chemical_Protein,ENSDARP00000006504,160,Chemical,Protein,STITCH
4,CIDm91758408,Chemical_Protein,ENSDARP00000014299,241,Chemical,Protein,STITCH


In [11]:
# # =============================================================================
# # Save as pickle for downstream use
# # =============================================================================

# stitch_chem_protein_raw.to_pickle(STITCH_PICKLE_PATH)
# print(f"Pickle saved to: {STITCH_PICKLE_PATH}")
# print(f"Total interactions: {len(stitch_chem_protein_raw):,}")

---
## 3. KG Triple Construction (Pickle → Annotated Triples)

Load the pre-processed STITCH pickle, clean chemical IDs (strip STITCH prefixes), map protein IDs to gene symbols via gProfiler, and annotate with descriptions.

In [12]:

chem_gene_df =stitch_chem_protein_raw

print(f"STITCH Chemical-Protein interactions loaded: {len(chem_gene_df):,}")

STITCH Chemical-Protein interactions loaded: 74,619,878


In [13]:
# =============================================================================
# Clean chemical IDs — strip STITCH prefixes and leading zeros
# =============================================================================

def clean_chemical_id(chem_id):
    """Remove STITCH-specific prefixes and leading zeros from chemical IDs."""
    return chem_id.lstrip('CIDs').lstrip('CIDm').lstrip('0')

# Note: DataFrame.applymap() is deprecated in newer pandas — should use .map() instead
chem_gene_df[['Head']] = chem_gene_df[['Head']].applymap(clean_chemical_id)

# Add KG metadata
chem_gene_df['relation'] = 'ChemicalEntity_Gene'
chem_gene_df['head_type'] = 'ChemicalEntity'
chem_gene_df['tail_type'] = 'Gene'
chem_gene_df['relation_type'] = np.nan
chem_gene_df['kg_source'] = 'STITCH'

# Map Ensembl Protein IDs to gene symbols using gProfiler
chem_gene_df['tail'] = chem_gene_df['Tail'].map(protein_to_gene_dict)

print(f"Chemical IDs cleaned, protein IDs mapped to gene symbols")
print(f"Mapped genes: {chem_gene_df['tail'].notna().sum():,}")
print(f"Unmapped (NaN): {chem_gene_df['tail'].isna().sum():,}")

/tmp/ipykernel_1661938/3202424894.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chem_gene_df[['Head']] = chem_gene_df[['Head']].applymap(clean_chemical_id)


Chemical IDs cleaned, protein IDs mapped to gene symbols
Mapped genes: 43,674,740
Unmapped (NaN): 30,945,138


---
## 4. Annotate, Filter, Validate, Deduplicate & Export

Map gene descriptions and chemical IUPAC names, filter unmapped entries, standardize schema, validate, deduplicate, and export.

In [14]:
# =============================================================================
# Map descriptions and filter unmapped entries
# =============================================================================

# Drop original 'Tail' column (Ensembl ID) — replaced by 'tail' (gene symbol)
# Note: DataFrame.drop() with inplace=True is planned for deprecation in future pandas
chem_gene_df.drop('Tail', axis=1, inplace=True)

# Map gene descriptions from gProfiler
chem_gene_df['tail_detail_name'] = chem_gene_df['tail'].map(gene_to_description_dict)

# Map chemical IUPAC names from PubChem
chem_gene_df['head_detail_name'] = chem_gene_df['Head'].map(pubchem_cid_to_iupac)

# Filter out entries with unmapped genes or descriptions
before_count = len(chem_gene_df)
chem_gene_df = chem_gene_df[~chem_gene_df['tail'].isna()]
chem_gene_df = chem_gene_df[~chem_gene_df['tail_detail_name'].isna()]
chem_gene_df = chem_gene_df[~chem_gene_df['head_detail_name'].isna()]
after_count = len(chem_gene_df)

print(f"Before filtering: {before_count:,}")
print(f"After filtering: {after_count:,}")
print(f"Removed: {before_count - after_count:,} triples with unmapped entities")

Before filtering: 74,619,878
After filtering: 29,386,948
Removed: 45,232,930 triples with unmapped entities


In [15]:
# =============================================================================
# Standardize columns and add remaining metadata
# =============================================================================

chem_gene_df.columns = chem_gene_df.columns.str.lower()
chem_gene_df['species'] = 'D.rerio'
chem_gene_df['head_id_is'] = 'Pubchem'
chem_gene_df['tail_id_is'] = 'NCBI_ID'
chem_gene_df['relation'] = 'ChemicalEntity_Gene'

# Remove duplicate columns (if any from mixed-case renaming)
cols = pd.Series(chem_gene_df.columns)
duplicates = cols[cols.duplicated()].unique()
for dup in duplicates:
    first_index = cols[cols == dup].index[0]
    chem_gene_df.drop(chem_gene_df.columns[first_index], axis=1, inplace=True)

# Handle duplicate 'tail' columns if present
tail_indices = [i for i, col in enumerate(chem_gene_df.columns) if col == 'tail']
if len(tail_indices) > 1:
    chem_gene_df.drop(chem_gene_df.columns[tail_indices[0]], axis=1, inplace=True)

print(f"Final columns: {list(chem_gene_df.columns)}")
print(f"Triples: {len(chem_gene_df):,}")

Final columns: ['head', 'combined_score', 'head_type', 'tail_type', 'relation_source', 'relation_type', 'kg_source', 'tail', 'tail_detail_name', 'head_detail_name', 'species', 'head_id_is', 'tail_id_is']
Triples: 29,386,948


In [18]:
chem_gene_df['relation'] = 'ChemicalEntity_Gene'


In [16]:
# =============================================================================
# Data quality validation
# =============================================================================

true_nan_count = chem_gene_df.isna().sum()
string_nan_count = chem_gene_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})

print("NaN validation summary:")
nan_summary

NaN validation summary:


,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
combined_score,0,0,0
head_type,0,0,0
tail_type,0,0,0
relation_source,0,0,0
relation_type,29386948,29386948,58773896
kg_source,0,0,0
tail,0,0,0
tail_detail_name,0,0,0
head_detail_name,0,0,0


In [19]:
# =============================================================================
# Deduplicate by grouping on (head, relation, tail)
# =============================================================================

group_cols = ['head', 'relation', 'tail']

def merge_sources(x):
    """Merge multiple kg_source values with '::' separator."""
    return '::'.join(sorted(set(x.dropna().astype(str))))

chem_gene_dedup = chem_gene_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Triples before deduplication: {len(chem_gene_df):,}")
print(f"Triples after deduplication: {len(chem_gene_dedup):,}")
print(f"Duplicates removed: {len(chem_gene_df) - len(chem_gene_dedup):,}")
chem_gene_dedup.head()

Triples before deduplication: 29,386,948
Triples after deduplication: 16,699,407
Duplicates removed: 12,687,541


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1,ChemicalEntity_Gene,ACADSB,ChemicalEntity,NaN,Gene,STITCH,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acyl-CoA dehydrogenase short/branched chain [S...,D.rerio
1,1,ChemicalEntity_Gene,ACY1,ChemicalEntity,NaN,Gene,STITCH,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,aminoacylase 1 [Source:HGNC Symbol;Acc:HGNC:177],D.rerio
2,1,ChemicalEntity_Gene,BPTF,ChemicalEntity,NaN,Gene,STITCH,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,bromodomain PHD finger transcription factor [S...,D.rerio
3,1,ChemicalEntity_Gene,BRPF3,ChemicalEntity,NaN,Gene,STITCH,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,bromodomain and PHD finger containing 3 [Sourc...,D.rerio
4,1,ChemicalEntity_Gene,CDY2B,ChemicalEntity,NaN,Gene,STITCH,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,chromodomain Y-linked 2B [Source:HGNC Symbol;A...,D.rerio


In [20]:
OUTPUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch_ZEBRA_CHEMICALENTITY_GENE.csv'

In [21]:
# =============================================================================
# Export final deduplicated Chemical-Gene KG triples
# =============================================================================

chem_gene_dedup.to_csv(OUTPUT_PATH, index=None)
print(f"Final output saved to: {OUTPUT_PATH}")
print(f"Total triples exported: {len(chem_gene_dedup):,}")

Final output saved to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch_ZEBRA_CHEMICALENTITY_GENE.csv
Total triples exported: 16,699,407


---
## Summary

This notebook processed STITCH Chemical–Protein interaction data for Zebrafish into standardized KG triples:

1. **Dictionary setup** — Loaded gProfiler (~29K protein→gene mappings) and PubChem (CID→IUPAC)
2. **Raw STITCH preprocessing** — Loaded ~74M raw interactions from TSV, cleaned protein IDs, saved pickle
3. **KG triple construction** — Cleaned chemical IDs, mapped proteins to genes via gProfiler
4. **Annotation & filtering** — Added gene descriptions and chemical names, filtered unmapped entries (~74M → ~33M)
5. **Deduplication** — Grouped by (head, relation, tail) for ~18.9M unique final triples

### Output Files

| File | Description |
|------|-------------|

| `stitch_ZEBRA_CHEMICALENTITY_GENE.csv` | Final: deduplicated relation-wise KG triples |